# D83 超参数敏感性总扫描(含 D82 的 β 双锚; 取代 d82_beta_sensitivity.ipynb)

统一协议: d73_AB 配置 + 同预算(50 轮 8192 步; C 组等总环境步)+ 单种子 + 每次只改一个参数 + byR 选择。
曲线中心 = d73_AB 前 50 轮 byR **123,561.5@44**(不另跑); β_r=0 端点复用 `abl_noroute_s0`。

| 参数 | 现值 | 扫描 |
|---|---|---|
| β_r routing anchor | 0.03 | 0(已有), 0.01, 0.1 |
| β_t timing anchor | 0.01 | 0, 0.005, 0.02 |
| lr | 5e-5 | 2e-5, 1e-4, 2e-4 |
| steps/iter | 8192×50 | 2048×200, 4096×100, 16384×25(等步) |
| promote-delay-k | 1.0 | 0.9, 1.1, 1.25 |
| fail_penalty | 50 | 25, 100, 200 |

17 个 run 全并行约占 13 核, 一晚绰绰有余。全部 cell 重跑安全(`--resume` 幂等)。


In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project
PROJ='/rds/homes/h/hxn886/ondemand/final_project'
PY='/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/bin/python'
PREFIX=(f'cd {PROJ} && env PYTHONUNBUFFERED=1 PYTHONPATH=. '
        f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MPLBACKEND=Agg {PY}')
print('OK')


## 预检(必须先过)


In [ ]:
# 预检: 确认服务器代码/ckpt 是新版(防止跑在过期拷贝上, 白烧机时)
import os
fails = []
def chk(cond, msg):
    print(('PASS ' if cond else 'FAIL ') + msg)
    if not cond: fails.append(msg)
env = open('derbysim/sim/env_dynamic.py').read()
s13 = open('scripts/13_train_ppo.py').read()
ppo = open('derbysim/learn/ppo.py').read()
chk('control_classes' in env and 'background' in env, 'env_dynamic 含 control_classes/background (D67)')
chk('_check_cycles' in env or 'bounce' in env, 'env_dynamic 含死锁环手术 (D73)')
chk('--fail-penalty' in s13 and '--promote-delay-k' in s13 and '--bc-anchor-beta' in s13, 'scripts/13 旗标齐全')
chk('bc_kind_anchor_kl' in ppo, 'ppo.py 含时机锚')
chk(os.path.exists('artifacts/reports/ppo_push_AB_s0_best.pt'), 'init ckpt 在位')
print(f"sizes: env_dynamic={os.path.getsize('derbysim/sim/env_dynamic.py')} (本地=57420), "
      f"scripts/13={os.path.getsize('scripts/13_train_ppo.py')} (本地=43738)")
if fails: raise SystemExit('预检失败! 先从本地同步: derbysim/ 整个包 + scripts/13_train_ppo.py, 再重跑本 cell')
print('预检通过, 可以发射')


## A. D82 β 双锚(5 run, 各约 35 分钟)


In [ ]:
import subprocess, os
os.makedirs('rl_logs', exist_ok=True)
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_br001_s0 --kl-beta 0.01 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 8192 --iters 50 --promote-delay-k 1.0 --fail-penalty 50 --kl-anchor schedule --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_br001_s0.log 2>&1 &'])
print('launched sens_br001_s0')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_br01_s0 --kl-beta 0.1 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 8192 --iters 50 --promote-delay-k 1.0 --fail-penalty 50 --kl-anchor schedule --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_br01_s0.log 2>&1 &'])
print('launched sens_br01_s0')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_bt0_s0 --kl-beta 0.03 --bc-anchor-beta 0 --lr 5e-5 --steps-per-iter 8192 --iters 50 --promote-delay-k 1.0 --fail-penalty 50 --kl-anchor schedule --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_bt0_s0.log 2>&1 &'])
print('launched sens_bt0_s0')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_bt0005_s0 --kl-beta 0.03 --bc-anchor-beta 0.005 --lr 5e-5 --steps-per-iter 8192 --iters 50 --promote-delay-k 1.0 --fail-penalty 50 --kl-anchor schedule --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_bt0005_s0.log 2>&1 &'])
print('launched sens_bt0005_s0')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_bt002_s0 --kl-beta 0.03 --bc-anchor-beta 0.02 --lr 5e-5 --steps-per-iter 8192 --iters 50 --promote-delay-k 1.0 --fail-penalty 50 --kl-anchor schedule --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_bt002_s0.log 2>&1 &'])
print('launched sens_bt002_s0')


## B. lr / 课程阈值 / 搁浅罚(6 run, 各约 35 分钟)


In [ ]:
import subprocess, os
os.makedirs('rl_logs', exist_ok=True)
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_lr2e5_s0 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 2e-5 --steps-per-iter 8192 --iters 50 --promote-delay-k 1.0 --fail-penalty 50 --kl-anchor schedule --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_lr2e5_s0.log 2>&1 &'])
print('launched sens_lr2e5_s0')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_lr1e4_s0 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 1e-4 --steps-per-iter 8192 --iters 50 --promote-delay-k 1.0 --fail-penalty 50 --kl-anchor schedule --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_lr1e4_s0.log 2>&1 &'])
print('launched sens_lr1e4_s0')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_pk11_s0 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 8192 --iters 50 --promote-delay-k 1.1 --fail-penalty 50 --kl-anchor schedule --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_pk11_s0.log 2>&1 &'])
print('launched sens_pk11_s0')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_pk125_s0 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 8192 --iters 50 --promote-delay-k 1.25 --fail-penalty 50 --kl-anchor schedule --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_pk125_s0.log 2>&1 &'])
print('launched sens_pk125_s0')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_fp25_s0 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 8192 --iters 50 --promote-delay-k 1.0 --fail-penalty 25 --kl-anchor schedule --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_fp25_s0.log 2>&1 &'])
print('launched sens_fp25_s0')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_fp100_s0 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 8192 --iters 50 --promote-delay-k 1.0 --fail-penalty 100 --kl-anchor schedule --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_fp100_s0.log 2>&1 &'])
print('launched sens_fp100_s0')


## C. 批量(等总环境步 409,600, 各约 1–2 小时)


In [ ]:
import subprocess, os
os.makedirs('rl_logs', exist_ok=True)
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_sp2048_s0 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 2048 --iters 200 --promote-delay-k 1.0 --fail-penalty 50 --kl-anchor schedule --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_sp2048_s0.log 2>&1 &'])
print('launched sens_sp2048_s0')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_sp4096_s0 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 4096 --iters 100 --promote-delay-k 1.0 --fail-penalty 50 --kl-anchor schedule --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_sp4096_s0.log 2>&1 &'])
print('launched sens_sp4096_s0')


## D. 补点(每面板 4 点、ours 居内; D83.1)

| run | 改动 | 目的 |
|---|---|---|
| sens_sp16384_s0 | steps 16384×25 轮(等总步) | 批量中心变内点 |
| sens_pk09_s0 | promote-delay-k 0.9 | 阈值中心变内点 |
| sens_lr2e4_s0 | lr 2e-4 | 高 lr 恶化侧 |
| sens_fp200_s0 | fail_penalty 200 | 六面板统一 4 点 |


In [ ]:
import subprocess, os
os.makedirs('rl_logs', exist_ok=True)
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_sp16384_s0 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 16384 --iters 25 --promote-delay-k 1.0 --fail-penalty 50 --kl-anchor schedule --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_sp16384_s0.log 2>&1 &'])
print('launched sens_sp16384_s0')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_pk09_s0 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 8192 --iters 50 --promote-delay-k 0.9 --fail-penalty 50 --kl-anchor schedule --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_pk09_s0.log 2>&1 &'])
print('launched sens_pk09_s0')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_lr2e4_s0 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 2e-4 --steps-per-iter 8192 --iters 50 --promote-delay-k 1.0 --fail-penalty 50 --kl-anchor schedule --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_lr2e4_s0.log 2>&1 &'])
print('launched sens_lr2e4_s0')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_fp200_s0 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 8192 --iters 50 --promote-delay-k 1.0 --fail-penalty 200 --kl-anchor schedule --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_fp200_s0.log 2>&1 &'])
print('launched sens_fp200_s0')


## 监控


In [ ]:
import glob, subprocess
for lg in sorted(glob.glob('rl_logs/sens_*.log')):
    out = subprocess.run(['tail','-n','2',lg], capture_output=True, text=True).stdout.strip()
    print('====', lg); print(out)


## 汇总(全部跑完后)


In [ ]:
import json, glob
def by_reward(f):
    evs = json.load(open(f))['evals']
    b = max(evs, key=lambda e: e['eval_episode_reward'])
    return b['iter'], b['eval_weighted_secondary_delay_s'], b['eval_completion_rate']
evs50 = [e for e in json.load(open('artifacts/reports/ppo_d73_AB_s0.json'))['evals'] if e['iter'] < 50]
b = max(evs50, key=lambda e: e['eval_episode_reward'])
print(f"{'center(d73_AB<50)':<20s}{b['iter']:>6d}{b['eval_weighted_secondary_delay_s']:>13,.1f}{b['eval_completion_rate']:>7.3f}")
for f in sorted(glob.glob('artifacts/reports/sens_*_s0.json')):
    it, d, c = by_reward(f)
    print(f"{f.split('/')[-1][:-5]:<20s}{it:>6d}{d:>13,.1f}{c:>7.3f}")
print('参考: abl_noroute_s0(β_r=0) 134,188.0; full(100轮) 123,035.5; fcfs 181,163.5')


## 完成后拷回本地

拷回 `artifacts/reports/sens_*_s0.json`(17 个)到本地同路径, 然后本地跑:
- `python scripts/43_beta_sensitivity.py` → F15(β 双面板)
- `python scripts/45_sensitivity_grid.py` → F17(六参数 2×3 大图; 43/44 的 F15/F16 仍可单独出)
两个脚本都容忍缺文件, 可边跑边看。
